In [ ]:
import pandas as pd
import json
import ast
import spacy
import networkx as nx
from tqdm import tqdm
from collections import defaultdict

!pip install neo4j
from neo4j import GraphDatabase

import os
#---Titel--
MIN_CO_PURCHASES = 3
NEO4J_URI = ""
NEO4J_USER = ""
NEO4J_PASSWORD = ""
BATCH_SIZE =  500

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 6.6 MB/s eta 0:00:00


In [ ]:
reviews_df = pd.read_csv('/content/electronics_reviews.csv')
metadata_df = pd.read_csv('/content/electronics_metadata.csv')

In [ ]:
reviews_df.head()

,rating,title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,date
0,3.0,Smells like gasoline! Going back!,First & most offensive: they reek of gasoline ...,B083NRGZMM,B083NRGZMM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1658185117948,0,True,2022-07-18 22:58:37.948
1,1.0,Didn’t work at all lenses loose/broken.,These didn’t work. Idk if they were damaged in...,B07N69T6TM,B07N69T6TM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1592678549731,0,True,2020-06-20 18:42:29.731
2,5.0,Excellent!,I love these. They even come with a carry case...,B01G8JO5F2,B01G8JO5F2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1523093017534,0,True,2018-04-07 09:23:37.534
3,5.0,Great laptop backpack!,I was searching for a sturdy backpack for scho...,B001OC5JKY,B001OC5JKY,AGGZ357AO26RQZVRLGU4D4N52DZQ,1290278495000,18,True,2010-11-20 18:41:35.000
4,5.0,Best Headphones in the Fifties price range!,I've bought these headphones three times becau...,B013J7WUGC,B07CJYMRWM,AG2L7H23R5LLKDKLBEF2Q3L2MVDA,1676601581238,0,True,2023-02-17 02:39:41.238


In [ ]:
reviews_df.shape

(10000, 10)

In [ ]:
metadata_df.head()

,main_category,title,average_rating,rating_number,features,description,price,store,categories,details,parent_asin
0,All Electronics,FS-1051 FATSHARK TELEPORTER V3 HEADSET,3.5,6,[],['Teleporter V3 The “Teleporter V3” kit sets a...,NaN,Fat Shark,['Electronics' 'Television & Video' 'Video Gla...,"{""Date First Available"": ""August 2, 2014"", ""Ma...",B00MCW7G9M
1,All Electronics,Ce-H22B12-S1 4Kx2K Hdmi 4Port,5.0,1,['UPC: 662774021904' 'Weight: 0.600 lbs'],['HDMI In - HDMI Out'],NaN,SIIG,['Electronics' 'Television & Video' 'Accessori...,"{""Product Dimensions"": ""0.83 x 4.17 x 2.05 inc...",B00YT6XQSE
2,Computers,Digi-Tatoo Decal Skin Compatible With MacBook ...,4.5,246,['WARNING: Please IDENTIFY MODEL NUMBER on the...,[],19.99,Digi-Tatoo,['Electronics' 'Computers & Accessories' 'Lapt...,"{""Brand"": ""Digi-Tatoo"", ""Color"": ""Fresh Marble...",B07SM135LS
3,AMAZON FASHION,NotoCity Compatible with Vivoactive 4 band 22m...,4.5,233,['☛NotoCity 22mm band is designed for Vivoacti...,[],9.99,NotoCity,"['Electronics' 'Wearable Technology' 'Clips, A...","{""Date First Available"": ""May 29, 2020"", ""Manu...",B089CNGZCW
4,Cell Phones & Accessories,Motorola Droid X Essentials Combo Pack,3.8,64,['New Droid X Essentials Combo Pack'\n 'Exclus...,['all Genuine High Quality Motorola Made Acces...,14.99,Verizon,['Electronics' 'Computers & Accessories'\n 'Co...,"{""Product Dimensions"": ""11.6 x 6.9 x 3.1 inche...",B004E2Z88O


In [ ]:
metadata_df.shape

(5000, 11)

In [ ]:
#-------------NLP---------
print("loading spacy model...")
try:
  nlp = spacy.load("en_core_web_sm")
  nlp.max_length = 1_000_000
except OSError:
  raise SystemExit(
        "❌ spaCy model not found.\n"
        "   Run: python -m spacy download en_core_web_sm"
    )

loading spacy model...


In [ ]:
def safe_parse_list(value) -> list:

  if isinstance(value, list):
    return value
  if isinstance(value, str):
    try:
      parsed = ast.literal_eval(value)
      return parsed if isinstance(parsed, list) else []
    except Exception:
      return [value] if value.strip() else []
  return []


In [ ]:
def rating_sentiment(rating:float) -> str:
  if rating >= 4:
    return 'positive'
  elif rating == 3:
    return 'neutral'
  else:
    return 'negative'

In [ ]:
def extract_noun_phrases(text:str, max_chars:int = 500) -> list[str]:
  if not isinstance(text, str) or not text.strip():
    return[]

  doc = nlp(text[:max_chars])
  phrases = []
  for chunk in doc.noun_chunks:
    phrase = chunk.text.strip().lower()
        # Filter noise: keep 2–5 word phrases, skip stopword-only chunks
    words = [t for t in chunk if not t.is_stop and not t.is_punct]
    if 1 <= len(phrase.split()) <= 5 and len(words) >= 1:
      phrases.append(phrase)

  return list(set(phrases))


In [ ]:
def extract_product_entities(meta_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    From metadata extract:
      - products   (asin, title, price, avg_rating)
      - brands     (name)
      - categories (name, level)
      - features   (name) + product→feature edges
    """
    print("\n🔍 Extracting product entities from metadata...")

    products   = []
    brands     = set()
    categories = set()
    features   = set()

    prod_brand_edges    = []
    prod_category_edges = []
    prod_feature_edges  = []

    for _, row in tqdm(meta_df.iterrows(), total=len(meta_df), desc="Products"):
        asin = row.get("asin") or row.get("parent_asin")
        if not asin:
            continue

        title = str(row.get("title", "")).strip()
        price = row.get("price")
        avg_rating = row.get("average_rating")

        products.append({
            "asin": asin,
            "title": title[:200],
            "price": price,
            "average_rating": avg_rating,
            "rating_number": row.get("rating_number"),
        })

        # Brand / Store
        brand = str(row.get("store", "")).strip()
        if brand and brand.lower() not in ("", "nan", "none"):
            brands.add(brand)
            prod_brand_edges.append({"asin": asin, "brand": brand})

        # Categories
        cats = safe_parse_list(row.get("categories", []))
        for cat in cats:
            cat = str(cat).strip()
            if cat:
                categories.add(cat)
                prod_category_edges.append({"asin": asin, "category": cat})

        # Features from bullet points
        feature_bullets = safe_parse_list(row.get("features", []))
        desc_text = " ".join(feature_bullets)
        noun_phrases = extract_noun_phrases(desc_text)
        for phrase in noun_phrases:
            features.add(phrase)
            prod_feature_edges.append({"asin": asin, "feature": phrase})

    products_df         = pd.DataFrame(products).drop_duplicates("asin")
    brands_df           = pd.DataFrame({"brand": list(brands)})
    categories_df       = pd.DataFrame({"category": list(categories)})
    features_df         = pd.DataFrame({"feature": list(features)})

    prod_brand_df       = pd.DataFrame(prod_brand_edges)
    prod_category_df    = pd.DataFrame(prod_category_edges)
    prod_feature_df     = pd.DataFrame(prod_feature_edges).drop_duplicates()

    print(f"   ✅ Products  : {len(products_df):,}")
    print(f"   ✅ Brands    : {len(brands_df):,}")
    print(f"   ✅ Categories: {len(categories_df):,}")
    print(f"   ✅ Features  : {len(features_df):,}")

    return (
        products_df, brands_df, categories_df, features_df,
        prod_brand_df, prod_category_df, prod_feature_df
    )


In [ ]:
products_df, brands_df, categories_df, features_df, prod_brand_df, prod_category_df, prod_feature_df  = extract_product_entities(metadata_df)


🔍 Extracting product entities from metadata...


Products: 100%|██████████| 5000/5000 [01:19<00:00, 63.07it/s]


   ✅ Products  : 5,000
   ✅ Brands    : 3,345
   ✅ Categories: 563
   ✅ Features  : 34,895


In [ ]:
def extract_review_entities(reviews_df: pd.DataFrame, valid_asins: set) -> tuple:
    """
    From reviews extract:
      - users
      - user→product REVIEWED edges (with rating + sentiment)
      - product→product CO_PURCHASED edges (shared reviewers)
    """
    print("\n🔍 Extracting review entities...")

    users = set()
    review_edges = []
    user_products = defaultdict(list)  # user_id → [asin, ...]

    for _, row in tqdm(reviews_df.iterrows(), total=len(reviews_df), desc="Reviews"):
        asin    = str(row.get("asin", "")).strip()
        user_id = str(row.get("user_id", "")).strip()
        rating  = row.get("rating")

        if not asin or not user_id or asin not in valid_asins:
            continue
        if user_id.lower() in ("", "nan", "none"):
            continue

        users.add(user_id)
        sentiment = rating_sentiment(float(rating)) if rating else "unknown"

        review_edges.append({
            "user_id": user_id,
            "asin": asin,
            "rating": rating,
            "sentiment": sentiment,
            "date": row.get("date", ""),
        })
        user_products[user_id].append(asin)

    # Co-purchased edges: products reviewed by the same user
    print("\n   Building co-purchased edges...")
    co_purchase_counts = defaultdict(int)
    for asins in tqdm(user_products.values(), desc="Co-purchase"):
        unique_asins = list(set(asins))
        for i in range(len(unique_asins)):
            for j in range(i + 1, len(unique_asins)):
                pair = tuple(sorted([unique_asins[i], unique_asins[j]]))
                co_purchase_counts[pair] += 1

    co_purchase_edges = [
        {"asin_1": a, "asin_2": b, "shared_users": count}
        for (a, b), count in co_purchase_counts.items()
        if count >= MIN_CO_PURCHASES
    ]

    users_df         = pd.DataFrame({"user_id": list(users)})
    review_edges_df  = pd.DataFrame(review_edges)
    co_purchase_df   = pd.DataFrame(co_purchase_edges)

    print(f"   ✅ Users           : {len(users_df):,}")
    print(f"   ✅ Review edges    : {len(review_edges_df):,}")
    print(f"   ✅ Co-purchase edges: {len(co_purchase_df):,}")

    return users_df, review_edges_df, co_purchase_df

In [ ]:
valid_asins = set(products_df["asin"].tolist())
users_df, review_edges_df, co_purchase_df = extract_review_entities(reviews_df, valid_asins)


🔍 Extracting review entities...


Reviews: 100%|██████████| 10000/10000 [00:00<00:00, 14541.17it/s]



   Building co-purchased edges...


Co-purchase: 100%|██████████| 15/15 [00:00<00:00, 118483.16it/s]


   ✅ Users           : 15
   ✅ Review edges    : 16
   ✅ Co-purchase edges: 0


In [ ]:
def build_network_graph(products_df, brands_df, categories_df, features_df,
    users_df, review_edges_df, co_purchase_df,
    prod_brand_df, prod_category_df, prod_feature_df) -> nx.Graph:

    print("\n🕸️  Building NetworkX Knowledge Graph...")

    G = nx.MultiDiGraph()

    for _, r in products_df.iterrows():
        G.add_node(r["asin"], type="product", title=r["title"],
                   price=r.get("price"), avg_rating=r.get("average_rating"))

    for _, r in brands_df.iterrows():
        G.add_node(r["brand"], type="brand")

    for _, r in categories_df.iterrows():
        G.add_node(r["category"], type="category")

    for _, r in features_df.iterrows():
        G.add_node(r["feature"], type="feature")

    for _, r in users_df.iterrows():
        G.add_node(r["user_id"], type="user")

    # Add edges
    for _, r in prod_brand_df.iterrows():
        G.add_edge(r["asin"], r["brand"], relation="MADE_BY")

    for _, r in prod_category_df.iterrows():
        G.add_edge(r["asin"], r["category"], relation="BELONGS_TO")

    for _, r in prod_feature_df.iterrows():
        G.add_edge(r["asin"], r["feature"], relation="HAS_FEATURE")

    for _, r in review_edges_df.iterrows():
        G.add_edge(r["user_id"], r["asin"], relation="REVIEWED",
                   rating=r.get("rating"), sentiment=r.get("sentiment"))

    for _, r in co_purchase_df.iterrows():
        G.add_edge(r["asin_1"], r["asin_2"], relation="CO_PURCHASED",
                   shared_users=r["shared_users"])

    print(f"   ✅ Nodes: {G.number_of_nodes():,}")
    print(f"   ✅ Edges: {G.number_of_edges():,}")
    return G

In [ ]:
G = build_network_graph(products_df, brands_df, categories_df, features_df,
    users_df, review_edges_df, co_purchase_df,
    prod_brand_df, prod_category_df, prod_feature_df)


🕸️  Building NetworkX Knowledge Graph...
   ✅ Nodes: 43,808
   ✅ Edges: 61,542


In [ ]:
def create_summery(output_dir,G):

    # Save graph summary
    summary = {
        "nodes": G.number_of_nodes(),
        "edges": G.number_of_edges(),
        "node_types": {
            "products":   len(products_df),
            "brands":     len(brands_df),
            "categories": len(categories_df),
            "features":   len(features_df),
            "users":      len(users_df),
        },
        "edge_types": {
            "MADE_BY":       len(prod_brand_df),
            "BELONGS_TO":    len(prod_category_df),
            "HAS_FEATURE":   len(prod_feature_df),
            "REVIEWED":      len(review_edges_df),
            "CO_PURCHASED":  len(co_purchase_df),
        }
    }
    with open(f"{output_dir}/kg_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print(f"\n💾 All files saved to: {output_dir}/")
    print(json.dumps(summary, indent=2))

create_summery("/content/",G)



💾 All files saved to: /content//
{
  "nodes": 43808,
  "edges": 61542,
  "node_types": {
    "products": 5000,
    "brands": 3345,
    "categories": 563,
    "features": 34895,
    "users": 15
  },
  "edge_types": {
    "MADE_BY": 4965,
    "BELONGS_TO": 4638,
    "HAS_FEATURE": 51923,
    "REVIEWED": 16,
    "CO_PURCHASED": 0
  }
}


In [ ]:
class Neo4jUploader:
    def __init__(self, uri: str, user: str, password: str):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        self.driver.verify_connectivity()
        print("✅ Connected to Neo4j Aura")

    def close(self):
        self.driver.close()

    def run(self, query: str, params: dict = {}):
        with self.driver.session() as session:
            session.run(query, params)

    def run_batch(self, query: str, rows: list):
        with self.driver.session() as session:
            session.run(query, {"rows": rows})

    # ── CONSTRAINTS ───────────────────────────────────────────────────────────

    def create_constraints(self):
        print("\n📐 Creating constraints...")
        constraints = [
            "CREATE CONSTRAINT IF NOT EXISTS FOR (p:Product)  REQUIRE p.asin    IS UNIQUE",
            "CREATE CONSTRAINT IF NOT EXISTS FOR (b:Brand)    REQUIRE b.name    IS UNIQUE",
            "CREATE CONSTRAINT IF NOT EXISTS FOR (c:Category) REQUIRE c.name    IS UNIQUE",
            "CREATE CONSTRAINT IF NOT EXISTS FOR (f:Feature)  REQUIRE f.name    IS UNIQUE",
            "CREATE CONSTRAINT IF NOT EXISTS FOR (u:User)     REQUIRE u.user_id IS UNIQUE",
        ]
        for c in constraints:
            self.run(c)
        print("   ✅ Done")

    # ── NODE LOADERS ──────────────────────────────────────────────────────────

    def load_products(self, df: pd.DataFrame):
        query = """
        UNWIND $rows AS row
        MERGE (p:Product {asin: row.asin})
        SET p.title          = row.title,
            p.price          = toFloat(row.price),
            p.average_rating = toFloat(row.average_rating),
            p.rating_number  = toInteger(row.rating_number)
        """
        self._batch_upload(df, query, "Products")

    def load_brands(self, df: pd.DataFrame):
        query = """
        UNWIND $rows AS row
        MERGE (b:Brand {name: row.brand})
        """
        self._batch_upload(df, query, "Brands")

    def load_categories(self, df: pd.DataFrame):
        query = """
        UNWIND $rows AS row
        MERGE (c:Category {name: row.category})
        """
        self._batch_upload(df, query, "Categories")

    def load_features(self, df: pd.DataFrame):
        query = """
        UNWIND $rows AS row
        MERGE (f:Feature {name: row.feature})
        """
        self._batch_upload(df, query, "Features")

    def load_users(self, df: pd.DataFrame):
        query = """
        UNWIND $rows AS row
        MERGE (u:User {user_id: row.user_id})
        """
        self._batch_upload(df, query, "Users")

    # ── EDGE LOADERS ──────────────────────────────────────────────────────────

    def load_product_brand_edges(self, df: pd.DataFrame):
        query = """
        UNWIND $rows AS row
        MATCH (p:Product {asin: row.asin})
        MATCH (b:Brand   {name: row.brand})
        MERGE (p)-[:MADE_BY]->(b)
        """
        self._batch_upload(df, query, "MADE_BY edges")

    def load_product_category_edges(self, df: pd.DataFrame):
        query = """
        UNWIND $rows AS row
        MATCH (p:Product  {asin: row.asin})
        MATCH (c:Category {name: row.category})
        MERGE (p)-[:BELONGS_TO]->(c)
        """
        self._batch_upload(df, query, "BELONGS_TO edges")

    def load_product_feature_edges(self, df: pd.DataFrame):
        query = """
        UNWIND $rows AS row
        MATCH (p:Product {asin: row.asin})
        MATCH (f:Feature {name: row.feature})
        MERGE (p)-[:HAS_FEATURE]->(f)
        """
        self._batch_upload(df, query, "HAS_FEATURE edges")

    def load_review_edges(self, df: pd.DataFrame):
        query = """
        UNWIND $rows AS row
        MATCH (u:User    {user_id: row.user_id})
        MATCH (p:Product {asin:    row.asin})
        MERGE (u)-[r:REVIEWED]->(p)
        SET r.rating    = toFloat(row.rating),
            r.sentiment = row.sentiment,
            r.date      = toString(row.date)
        """
        self._batch_upload(df, query, "REVIEWED edges")

    def load_co_purchased_edges(self, df: pd.DataFrame):
        query = """
        UNWIND $rows AS row
        MATCH (a:Product {asin: row.asin_1})
        MATCH (b:Product {asin: row.asin_2})
        MERGE (a)-[r:CO_PURCHASED]-(b)
        SET r.shared_users = toInteger(row.shared_users)
        """
        self._batch_upload(df, query, "CO_PURCHASED edges")

    # ── BATCH HELPER ──────────────────────────────────────────────────────────

    def _batch_upload(self, df: pd.DataFrame, query: str, label: str):
        # Convert NaN → None so Neo4j doesn't choke on float('nan')
        df = df.where(pd.notna(df), other=None)
        rows = df.to_dict(orient="records")
        total = len(rows)

        with tqdm(total=total, desc=f"   ⬆ {label}", unit="rows") as pbar:
            for i in range(0, total, BATCH_SIZE):
                batch = rows[i : i + BATCH_SIZE]
                self.run_batch(query, batch)
                pbar.update(len(batch))

        print(f"   ✅ {label}: {total:,} rows uploaded")

    # ── VERIFY ────────────────────────────────────────────────────────────────

    def verify(self):
        print("\n🔍 Verifying counts in Neo4j Aura...")
        checks = [
            ("Product nodes",      "MATCH (n:Product)           RETURN count(n) AS n"),
            ("Brand nodes",        "MATCH (n:Brand)             RETURN count(n) AS n"),
            ("Category nodes",     "MATCH (n:Category)          RETURN count(n) AS n"),
            ("Feature nodes",      "MATCH (n:Feature)           RETURN count(n) AS n"),
            ("User nodes",         "MATCH (n:User)              RETURN count(n) AS n"),
            ("MADE_BY edges",      "MATCH ()-[:MADE_BY]->()     RETURN count(*) AS n"),
            ("BELONGS_TO edges",   "MATCH ()-[:BELONGS_TO]->()  RETURN count(*) AS n"),
            ("HAS_FEATURE edges",  "MATCH ()-[:HAS_FEATURE]->() RETURN count(*) AS n"),
            ("REVIEWED edges",     "MATCH ()-[:REVIEWED]->()    RETURN count(*) AS n"),
            ("CO_PURCHASED edges", "MATCH ()-[:CO_PURCHASED]-() RETURN count(*) AS n"),
        ]
        with self.driver.session() as session:
            for label, query in checks:
                result = session.run(query).single()
                print(f"   {label:<25} {result['n']:>8,}")


# ─── PUBLIC ENTRY POINT ───────────────────────────────────────────────────────

def upload_all(
    uri:           str,
    user:          str,
    password:      str,
    products_df:   pd.DataFrame,
    brands_df:     pd.DataFrame,
    categories_df: pd.DataFrame,
    features_df:   pd.DataFrame,
    users_df:      pd.DataFrame,
    prod_brand_df: pd.DataFrame,
    prod_cat_df:   pd.DataFrame,
    prod_feat_df:  pd.DataFrame,
    reviews_df:    pd.DataFrame,
    co_pur_df:     pd.DataFrame,
):
    """
    Upload all KG DataFrames to Neo4j Aura.
    Accepts DataFrames directly — no CSV files required.
    """
    print("🚀 Starting upload to Neo4j Aura...")
    print(f"   URI: {uri}\n")

    uploader = Neo4jUploader(uri, user, password)

    try:
        uploader.create_constraints()

        print("\n🟦 Uploading nodes...")
        uploader.load_products(products_df)
        uploader.load_brands(brands_df)
        uploader.load_categories(categories_df)
        uploader.load_features(features_df)
        uploader.load_users(users_df)

        print("\n🟧 Uploading edges...")
        uploader.load_product_brand_edges(prod_brand_df)
        uploader.load_product_category_edges(prod_cat_df)
        uploader.load_product_feature_edges(prod_feat_df)
        uploader.load_review_edges(reviews_df)
        uploader.load_co_purchased_edges(co_pur_df)

        uploader.verify()

        print("\n✅ All data uploaded successfully!")
        print("   Explore your graph: MATCH (n) RETURN n LIMIT 50")

    except Exception as e:
        print(f"\n❌ Upload failed: {e}")
        raise

    finally:
        uploader.close()


# ─── STANDALONE SCRIPT MODE ───────────────────────────────────────────────────

if __name__ == "__main__":
    import sys
    if "<your-instance>" in NEO4J_URI:
        print(
            "❌  Fill in NEO4J_URI and NEO4J_PASSWORD at the top of this file,\n"
            "    or call upload_all() directly from your notebook.\n"
            "    Aura console: https://console.neo4j.io\n"
        )
        sys.exit(1)

In [ ]:
upload_all(
    uri           = "",
    user          = "",
    password      = "",
    products_df   = products_df,
    brands_df     = brands_df,
    categories_df = categories_df,
    features_df   = features_df,
    users_df      = users_df,
    prod_brand_df = prod_brand_df,
    prod_cat_df   = prod_category_df,
    prod_feat_df  = prod_feature_df,
    reviews_df    = review_edges_df,
    co_pur_df     = co_purchase_df,
)

🚀 Starting upload to Neo4j Aura...
   URI: neo4j+s://7d5583fc.databases.neo4j.io

✅ Connected to Neo4j Aura

📐 Creating constraints...
   ✅ Done

🟦 Uploading nodes...


   ⬆ Products: 100%|██████████| 5000/5000 [00:04<00:00, 1180.34rows/s]


   ✅ Products: 5,000 rows uploaded


   ⬆ Brands: 100%|██████████| 3345/3345 [00:01<00:00, 1837.08rows/s]


   ✅ Brands: 3,345 rows uploaded


...

   ⬆ REVIEWED edges: 100%|██████████| 16/16 [00:00<00:00, 48.14rows/s]


   ✅ REVIEWED edges: 16 rows uploaded


   ⬆ CO_PURCHASED edges: 0rows [00:00, ?rows/s]


   ✅ CO_PURCHASED edges: 0 rows uploaded

🔍 Verifying counts in Neo4j Aura...
   Product nodes                5,000
   Brand nodes                  3,345
   Category nodes                 563
   Feature nodes               34,895
   User nodes                      15
   MADE_BY edges                4,965
   BELONGS_TO edges             4,638
   HAS_FEATURE edges           51,923
   REVIEWED edges                  16


   CO_PURCHASED edges               0

✅ All data uploaded successfully!
   Explore your graph: MATCH (n) RETURN n LIMIT 50
